In [10]:
import pandas as pd
import pickle

df = pd.read_csv("../data/raw/train.csv")

X = df.drop("SalePrice", axis=1)
y = df["SalePrice"]

# Load preprocessor
with open("../models/preprocessor.pkl", "rb") as f:
    preprocessor = pickle.load(f)

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
) 
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline

models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "Random Forest": RandomForestRegressor(),
    "Gradient Boosting": GradientBoostingRegressor()
}

pipelines = {}

for name, model in models.items():
    pipelines[name] = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
for name, pipeline in pipelines.items():
    print(f"Training {name}...")
    pipeline.fit(X_train, y_train)
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

results = []

for name, pipeline in pipelines.items():
    y_pred = pipeline.predict(X_test)
    
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    results.append([name, mae, rmse, r2])

results_df = pd.DataFrame(results, columns=["Model", "MAE", "RMSE", "R2"])
results_df.sort_values(by="RMSE")
best_model_name = results_df.sort_values(by="RMSE").iloc[0]["Model"]
print("Best Model:", best_model_name)
import os
import pickle

os.makedirs("../models", exist_ok=True)

best_pipeline = pipelines[best_model_name]

with open("../models/model.pkl", "wb") as f:
    pickle.dump(best_pipeline, f)

print("✅ Model saved successfully!")
with open("../models/model.pkl", "rb") as f:
    loaded_model = pickle.load(f)

sample = X_test.iloc[:1]
prediction = loaded_model.predict(sample)

print("Prediction:", prediction)
selected_features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "TotalBsmtSF"
]
X_simple = df[selected_features]
y = df["SalePrice"]
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X_train, X_test, y_train, y_test = train_test_split(
    X_simple, y, test_size=0.2, random_state=42
)

model_simple = LinearRegression()
model_simple.fit(X_train, y_train)
import pickle
import os

os.makedirs("../models", exist_ok=True)

with open("../models/simple_model.pkl", "wb") as f:
    pickle.dump(model_simple, f)

Training Linear Regression...
Training Ridge...
Training Lasso...
Training Random Forest...
Training Gradient Boosting...
Best Model: Gradient Boosting
✅ Model saved successfully!
Prediction: [144778.80945601]
